In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fredapi import Fred
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.stats import t as t_dist
import io
import os

# --- Settings ---
plt.style.use('fivethirtyeight')
INITIAL_INVESTMENT = 100000
# Default dates; the start date will be updated via widget.
START_DATE = "2009-12-31"
END_DATE = "2024-01-01"
fred = Fred(api_key='169131e6ef793368ca95adf00fec940a')

# --- Helper Functions ---
def fetch_ticker(ticker, start_date=None, end_date=None):
    """Fetch daily 'Close' prices for a given ticker."""
    if start_date is None:
        start_date = START_DATE
    if end_date is None:
        end_date = END_DATE

    try:
        data = yf.download(ticker, start=start_date, end=end_date)["Close"]
        if data.empty:
            raise ValueError(f"No data for {ticker}")
        return data
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None

def compute_inflation_adjustment(prices, cpi_series):
    """Return inflation-adjusted prices, normalized prices (to 1 at start), and adjustment factor."""
    base_cpi = cpi_series.iloc[0]
    adjustment = base_cpi / cpi_series
    adj_prices = prices * adjustment
    norm_prices = adj_prices / adj_prices.iloc[0]
    return adj_prices, norm_prices, adjustment

def max_drawdown(series):
    """Calculate maximum drawdown from a return series."""
    cum = (1 + series).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min()

def compute_var_t(returns, alpha=0.05):
    """Compute VaR at the alpha level using the Student's t-distribution."""
    mean = returns.mean()
    std = returns.std()
    df = len(returns) - 1
    q = t_dist.ppf(alpha, df)  # q is negative
    var = -(mean + std * q)
    return var

def compute_recovery_time(price_series):
    """Compute recovery time (in months) from the worst drawdown event."""
    cum = price_series / INITIAL_INVESTMENT
    peak = cum.cummax()
    dd = (cum - peak) / peak
    worst_date = dd.idxmin()
    prior_peak = cum.loc[:worst_date].max()
    recovery = cum.loc[worst_date:].loc[cum.loc[worst_date:] >= prior_peak]
    if not recovery.empty:
        recovery_date = recovery.index[0]
        return (recovery_date - worst_date).days / 30.0
    else:
        return np.nan

def analyze_portfolio(tickers, weight_dict, start_date):
    """
    Given a list of tickers and a weight dictionary, fetch data, apply CPI adjustment,
    build the portfolio, and compute performance metrics.
    Returns a dict with keys: "price", "returns", "metrics", "df".
    """
    data_dict = {}
    for t in tickers:
        s = fetch_ticker(t, start_date=start_date)
        if s is None:
            print(f"Data unavailable for {t}.")
            return None
        data_dict[t] = s
    df = pd.concat(data_dict, axis=1)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.dropna(inplace=True)

    try:
        cpi_series = fred.get_series("CPIAUCSL", start_date, END_DATE).resample("D").ffill()
    except Exception as e:
        print("Error fetching CPI:", e)
        return None
    cpi_series = cpi_series.reindex(df.index, method='ffill')
    base_cpi = cpi_series.iloc[0]
    df["CPI_Adjustment"] = base_cpi / cpi_series

    for t in tickers:
        price = df[t]
        if isinstance(price, pd.DataFrame):
            price = price.squeeze()
        adj, norm, _ = compute_inflation_adjustment(price, cpi_series)
        df[t + "_Real"] = adj
        df[t + "_norm"] = norm

    port_series = INITIAL_INVESTMENT * sum(
        weight_dict[t] * df[t + "_norm"] for t in tickers
    )
    df["Portfolio"] = port_series
    returns = df["Portfolio"].pct_change().fillna(0)

    annual_factor = 252
    ann_return = returns.mean() * annual_factor
    vol = returns.std() * np.sqrt(annual_factor)
    dd = max_drawdown(returns)
    sharpe = ann_return / vol if vol != 0 else np.nan
    var_t = compute_var_t(returns, alpha=0.05)
    recov_time = compute_recovery_time(port_series)

    metrics = {
        "Annual Return": ann_return,
        "Volatility": vol,
        "Max Drawdown": dd,
        "Sharpe Ratio": sharpe,
        "VaR (5%)": var_t,
        "Recovery Time (months)": recov_time
    }
    return {"price": port_series, "returns": returns, "metrics": metrics, "df": df}

# --- Interactive Widgets for Custom Portfolio Creation ---
available_custom = ["AAPL", "MSFT", "GOOG", "AMZN", "TSLA", "NFLX", "NVDA", "META", "YPF", "GGAL", "BMA", "PAM", "TGS", "IRS", "TEO", "CRESY", "BBAR", "MELI"]
custom_ticker_selector = widgets.SelectMultiple(
    options=available_custom,
    value=["AAPL", "MSFT", "GOOG"],
    description="Tickers:",
    layout=widgets.Layout(height='200px')
)
portfolio_name_input = widgets.Text(
    value="Custom1",
    description="Portfolio Name:"
)
custom_weight_method = widgets.Dropdown(
    options=["Equally Weighted", "Custom Weighted"],
    value="Equally Weighted",
    description="Weighting:"
)
custom_weight_box = widgets.VBox([])
custom_weight_inputs = {}

def update_custom_weight_inputs(*args):
    if custom_weight_method.value == "Custom Weighted":
        inputs = []
        for t in custom_ticker_selector.value:
            if t not in custom_weight_inputs:
                custom_weight_inputs[t] = widgets.FloatText(
                    value=0.0,
                    description=t+":",
                    step=0.1,
                    style={'description_width': 'initial'}
                )
            inputs.append(custom_weight_inputs[t])
        custom_weight_box.children = inputs
    else:
        custom_weight_box.children = []

custom_ticker_selector.observe(update_custom_weight_inputs, names='value')
custom_weight_method.observe(update_custom_weight_inputs, names='value')

add_portfolio_button = widgets.Button(
    description="Add Custom Portfolio", button_style="info"
)
custom_output = widgets.Output()

# --- Widget for Filtering Start Date ---
start_date_picker = widgets.DatePicker(
    description="Start Date:",
    value=pd.to_datetime("2009-12-31")
)

# Initialize with current date value
current_start_date = start_date_picker.value.strftime("%Y-%m-%d")

# --- Storage for portfolios ---
primary_portfolios = {}
custom_portfolios = {}

def initialize_primary_portfolios():
    """Initialize the primary portfolios."""
    start_date = start_date_picker.value.strftime("%Y-%m-%d")

    # Fetch and prepare primary tickers data
    primary_tickers = ["SPY", "LQD", "IEF", "GLD"]
    primary_data = {}
    for t in primary_tickers:
        s = fetch_ticker(t, start_date=start_date)
        if s is not None:
            primary_data[t] = s
    primary_df = pd.concat(primary_data, axis=1)
    primary_df.columns = primary_tickers
    primary_df.dropna(inplace=True)

    try:
        cpi_series = fred.get_series("CPIAUCSL", start_date, END_DATE).resample("D").ffill()
    except Exception as e:
        print("Error fetching CPI:", e)
        return {}

    cpi_series = cpi_series.reindex(primary_df.index, method='ffill')
    base_cpi = cpi_series.iloc[0]
    primary_df["CPI_Adjustment"] = base_cpi / cpi_series

    for t in primary_tickers:
        adj, norm, _ = compute_inflation_adjustment(primary_df[t], cpi_series)
        primary_df[t+"_Real"] = adj
        primary_df[t+"_norm"] = norm

    portfolios = {}
    # Cash Portfolio (inflation-adjusted cash)
    cash_series = INITIAL_INVESTMENT * primary_df["CPI_Adjustment"]
    portfolios["Cash"] = {"price": cash_series, "returns": cash_series.pct_change().fillna(0), "df": primary_df}

    # 100% SPY Portfolio
    spy_series = INITIAL_INVESTMENT * primary_df["SPY_norm"]
    portfolios["100% SPY"] = {"price": spy_series, "returns": spy_series.pct_change().fillna(0), "df": primary_df}

    # Multi-Asset Portfolio: 50% SPY, 20% LQD, 20% IEF, 10% GLD
    weights_primary = {"SPY": 0.5, "LQD": 0.2, "IEF": 0.2, "GLD": 0.1}
    multi_series = INITIAL_INVESTMENT * (
        weights_primary["SPY"] * primary_df["SPY_norm"] +
        weights_primary["LQD"] * primary_df["LQD_norm"] +
        weights_primary["IEF"] * primary_df["IEF_norm"] +
        weights_primary["GLD"] * primary_df["GLD_norm"]
    )
    portfolios["Multi-Asset"] = {
        "price": multi_series,
        "returns": multi_series.pct_change().fillna(0),
        "weights": weights_primary,
        "df": primary_df
    }

    return portfolios

def on_add_portfolio_clicked(b):
    with custom_output:
        clear_output()
        name = portfolio_name_input.value.strip()
        if name == "":
            print("Please enter a portfolio name.")
            return
        tickers = list(custom_ticker_selector.value)
        if not tickers:
            print("Please select at least one ticker.")
            return
        if custom_weight_method.value == "Equally Weighted":
            weight_dict = {t: 1/len(tickers) for t in tickers}
        else:
            raw = {t: custom_weight_inputs[t].value for t in tickers}
            total = sum(raw.values())
            if total == 0:
                print("Weights cannot sum to 0.")
                return
            weight_dict = {t: raw[t]/total for t in tickers}
        custom_portfolios[name] = {"tickers": tickers, "weights": weight_dict}
        print(f"Custom portfolio '{name}' added.")
        update_portfolio_selector()

add_portfolio_button.on_click(on_add_portfolio_clicked)

# --- Interactive Widgets for Combined Analysis ---
def get_all_portfolio_names():
    return list(primary_portfolios.keys()) + list(custom_portfolios.keys())

portfolio_selector = widgets.SelectMultiple(
    options=[],  # Will be populated after initialization
    value=[],
    description="Portfolios:"
)

run_analysis_button = widgets.Button(
    description="Run Combined Analysis", button_style="success"
)
analysis_output = widgets.Output()

def update_portfolio_selector():
    """Update the portfolio selector with current portfolio names."""
    names = get_all_portfolio_names()
    portfolio_selector.options = names
    if names:  # Select all by default
        portfolio_selector.value = names

def fig_to_excel(workbook, worksheet, fig, cell):
    """Save a matplotlib figure to an Excel worksheet."""
    imgdata = io.BytesIO()
    fig.savefig(imgdata, format='png')
    imgdata.seek(0)

    worksheet.insert_image(cell, "", {'image_data': imgdata})

def run_combined_analysis(b):
    with analysis_output:
        clear_output()

        # Get the current date from the picker and update global
        current_start_date = start_date_picker.value.strftime("%Y-%m-%d")

        # Re-initialize primary portfolios with the updated date
        global primary_portfolios
        primary_portfolios = initialize_primary_portfolios()

        selected = list(portfolio_selector.value)
        if not selected:
            print("Please select at least one portfolio.")
            return

        results = {}
        # Collect fixed (primary) portfolios
        for name in selected:
            if name in primary_portfolios:
                results[name] = primary_portfolios[name]
            elif name in custom_portfolios:
                defn = custom_portfolios[name]
                analysis = analyze_portfolio(defn["tickers"], defn["weights"], current_start_date)
                if analysis is None:
                    print(f"Analysis failed for {name}.")
                    continue
                results[name] = analysis

        if not results:
            print("No valid portfolios to analyze.")
            return

        # Prepare combined cumulative performance DataFrame and metrics list
        combined_ts = pd.DataFrame()
        returns_df = pd.DataFrame()
        drawdown_df = pd.DataFrame()
        metrics_list = []

        # Create a grid of subplots
        fig = plt.figure(figsize=(14, 10))
        grid = plt.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

        # Plot 1: Cumulative Performance (top-left)
        ax1 = fig.add_subplot(grid[0, 0])
        for p, data in results.items():
            ts = data["price"]
            cum_perf = ts / INITIAL_INVESTMENT
            combined_ts[p] = cum_perf
            ax1.plot(ts.index, cum_perf, label=p)
            ret = data["returns"]
            returns_df[p] = ret
            ann_return = ret.mean() * 252
            vol = ret.std() * np.sqrt(252)
            dd = max_drawdown(ret)
            sharpe = ann_return / vol if vol != 0 else np.nan
            var_t = compute_var_t(ret, alpha=0.05)
            recov = compute_recovery_time(ts)
            metrics_list.append({
                "Portfolio": p,
                "Annual Return": ann_return,
                "Volatility": vol,
                "Max Drawdown": dd,
                "Sharpe Ratio": sharpe,
                "VaR (5%)": var_t,
                "Recovery Time (months)": recov
            })
        ax1.set_title("Cumulative Performance")
        ax1.set_xlabel("Date")
        ax1.set_ylabel("Multiple of Initial Investment")
        ax1.legend(loc='upper left', fontsize='small')
        ax1.grid(True)

        # Plot 2: Distribution of Daily Returns (top-right)
        ax2 = fig.add_subplot(grid[0, 1])
        for p, data in results.items():
            ret = data["returns"]
            ax2.hist(ret, bins=50, alpha=0.5, label=p, density=True)
        ax2.set_title("Distribution of Daily Returns")
        ax2.set_xlabel("Daily Return")
        ax2.set_ylabel("Density")
        ax2.legend(loc='upper left', fontsize='small')
        ax2.grid(True)

        # Plot 3: Underwater (drawdown) plot (bottom-left)
        ax3 = fig.add_subplot(grid[1, 0])
        for p, data in results.items():
            ret = data["returns"]
            cum = (1+ret).cumprod()
            peak = cum.cummax()
            dd_series = (cum - peak) / peak
            drawdown_df[p] = dd_series
            ax3.plot(cum.index, dd_series, label=p)
        ax3.set_title("Underwater (Drawdown) Plot")
        ax3.set_xlabel("Date")
        ax3.set_ylabel("Drawdown")
        ax3.legend(loc='lower left', fontsize='small')
        ax3.grid(True)

        # Plot 4: Asset allocation donut charts (bottom-right)
        ax4 = fig.add_subplot(grid[1, 1])

        # Show donut chart for first portfolio
        if selected:
            p = selected[0]  # Just show the first selected portfolio's allocation
            if p == "Cash":
                contrib = {"Cash": 1.0}
            elif p == "100% SPY":
                contrib = {"SPY": 1.0}
            elif p == "Multi-Asset":
                contrib = results[p].get("weights", {"SPY": 0.5, "LQD": 0.2, "IEF": 0.2, "GLD": 0.1})
            else:
                defn = custom_portfolios[p]
                contrib = defn["weights"]

            labels = list(contrib.keys())
            sizes = [contrib[k] for k in labels]

            # Draw donut chart directly
            wedges, texts, autotexts = ax4.pie(
                sizes,
                labels=labels,
                autopct='%1.1f%%',
                startangle=90,
                wedgeprops={'width': 0.4}
            )
            ax4.set_title(f"Asset Allocation: {p}")
            ax4.axis('equal')

        plt.tight_layout()

        # Create metrics DataFrame
        metrics_df = pd.DataFrame(metrics_list)
        metrics_df_formatted = metrics_df.copy()
        metrics_df_formatted["Annual Return"] = metrics_df["Annual Return"].map("{:.2%}".format)
        metrics_df_formatted["Volatility"] = metrics_df["Volatility"].map("{:.2%}".format)
        metrics_df_formatted["Max Drawdown"] = metrics_df["Max Drawdown"].map("{:.2%}".format)
        metrics_df_formatted["Sharpe Ratio"] = metrics_df["Sharpe Ratio"].map("{:.2f}".format)
        metrics_df_formatted["VaR (5%)"] = metrics_df["VaR (5%)"].map("{:.2%}".format)
        metrics_df_formatted["Recovery Time (months)"] = metrics_df["Recovery Time (months)"].map(lambda x: f"{x:.1f}" if pd.notnull(x) else "N/A")

        # Show the metrics table
        print("Performance Metrics:")
        print(metrics_df_formatted.to_string(index=False))

        # Display the figure
        plt.show()

        # Create individual allocation charts for all portfolios
        allocation_data = {}
        allocation_figs = {}

        for p in selected:
            if p == "Cash":
                allocation_data[p] = {"Cash": 1.0}
            elif p == "100% SPY":
                allocation_data[p] = {"SPY": 1.0}
            elif p == "Multi-Asset":
                allocation_data[p] = results[p].get("weights", {"SPY": 0.5, "LQD": 0.2, "IEF": 0.2, "GLD": 0.1})
            else:
                allocation_data[p] = custom_portfolios[p]["weights"]

            # Create a separate allocation figure for each portfolio
            f = plt.figure(figsize=(6, 6))
            ax = f.add_subplot(111)
            labels = list(allocation_data[p].keys())
            sizes = [allocation_data[p][k] for k in labels]
            ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, wedgeprops={'width': 0.4})
            ax.set_title(f"Asset Allocation: {p}")
            ax.axis('equal')
            allocation_figs[p] = f
            plt.close(f)  # Don't display these individual charts now

        # Save all data to Excel
        with pd.ExcelWriter("combined_portfolio_analysis.xlsx", engine="xlsxwriter") as writer:
            workbook = writer.book

            # Sheet 1: Main plots
            worksheet_plots = workbook.add_worksheet("Analysis Charts")
            fig_to_excel(workbook, worksheet_plots, fig, 'A1')

            # Sheet 2: Performance Metrics
            metrics_df.to_excel(writer, sheet_name="Performance Metrics", index=False)

            # Sheet 3: Cumulative Performance Data
            combined_ts.to_excel(writer, sheet_name="Cumulative Performance")

            # Sheet 4: Daily Returns Data
            returns_df.to_excel(writer, sheet_name="Daily Returns")

            # Sheet 5: Drawdown Data
            drawdown_df.to_excel(writer, sheet_name="Drawdowns")

            # Sheet 6: Allocation data and charts
            allocation_worksheet = workbook.add_worksheet("Allocations")

            # Create a DataFrame from allocation data for Excel
            allocation_df = pd.DataFrame({p: pd.Series(allocation_data[p]) for p in selected})
            allocation_df = allocation_df.fillna(0)
            allocation_df.to_excel(writer, sheet_name="Allocation Data")

            # Add individual allocation charts
            row = 0
            for i, (p, fig) in enumerate(allocation_figs.items()):
                # Layout: 2 charts per row
                col = 0 if i % 2 == 0 else 8
                if i % 2 == 0 and i > 0:
                    row += 15  # Move down for next row of charts
                fig_to_excel(workbook, allocation_worksheet, fig, f'{chr(65+col)}{row+1}')

        print("Analysis complete! All data and charts saved to 'combined_portfolio_analysis.xlsx'")

# Initialize and set up the UI
def initialize_ui():
    # Initialize primary portfolios with default date
    global primary_portfolios
    primary_portfolios = initialize_primary_portfolios()

    # Update portfolio selector
    update_portfolio_selector()

    # Set up the button callback
    run_analysis_button.on_click(run_combined_analysis)

    # Display the UI components
    display(widgets.VBox([
        portfolio_name_input,
        custom_ticker_selector,
        custom_weight_method,
        custom_weight_box,
        add_portfolio_button,
        custom_output
    ]))

    display(widgets.VBox([
        start_date_picker,
        portfolio_selector,
        run_analysis_button,
        analysis_output
    ]))

# Run initialization
initialize_ui()


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
